# Pooled current-sheet thickness across the proton-beta scan

The thickness of every current sheet, pooled over several snapshots of each run, with a lognormal fit:
`pta.sheets.pooled_thickness` does the filtering, detection, measurement (M1, M2, M3), pooling and fitting, and
caches the filtered fields. This notebook only says which runs, which snapshots and which filter.

It replaces the ten copies of that pipeline the notebook used to hold (one per figure).

In [ ]:
import csv

import matplotlib.pyplot as plt

from pta import config
from pta.plotting import use_style
from pta.sheets import pooled_thickness

use_style()
RESULTS = config.results_dir("cs_identification", "PooledThicknessProtonBeta")

In [ ]:
BETAS_P = [0.0625, 0.25, 1, 4, 16]    # beta_e = 1, B_rms = 0.24 (the beta_scan layout in pta.toml)
BETA_E = 1
BRMS = 0.24
TIMES = range(200, 250, 5)            # the snapshots pooled for each run

# the filter: k = (k d_i where the density SNR drops to 3) / 3, per run (quality_checks/SNR)
K_CUTOFF = {0.0625: 38.5105 / 3, 0.25: 20.0 / 3, 1: 14.1252 / 3, 4: 8.0 / 3, 16: 4.3192 / 3}
XI = 1.6

METHODS = ("m1", "m2", "m3")          # "m3" alone for a single panel
N_BOOTSTRAP = 1000

In [ ]:
summary = []
for beta_p in BETAS_P:
    pooled = pooled_thickness(beta_p=beta_p, beta_e=BETA_E, brms=BRMS, times=TIMES,
                              k=K_CUTOFF[beta_p], Xi=XI, method=METHODS, bootstrap=N_BOOTSTRAP)
    folder = config.results_dir("cs_identification", "PooledThicknessProtonBeta", f"beta_p{beta_p}")
    figure_path, array_path = pooled.save(folder, f"pooled_thickness_beta_p{beta_p}")
    print(f"beta_p = {beta_p}: {pooled.sheets_per_time}\n  -> {figure_path}")
    plt.show()
    summary += [dict(run=f"beta_p{beta_p}", **row) for row in pooled.statistics]

In [ ]:
# One row per run and method
columns = ["run", "method", "n_snapshots", "n", "median_lognorm", "sigma_lognorm", "median_se", "ci_lo", "ci_hi",
           "mean", "median_emp"]
with open(RESULTS / "pooled_summary.csv", "w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=columns, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(summary)
print(f"saved {RESULTS / 'pooled_summary.csv'}")